# Tier 04 — Predict & Quantify (the crux)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/f-inverse/jammi-ai/blob/py-v0.49.1/cookbook/notebooks/book/04-predict/predict.ipynb)

Built from [`cookbook/book/chapters/04-predict/predict.qmd`](https://github.com/f-inverse/jammi-ai/blob/main/cookbook/book/chapters/04-predict/predict.qmd). Run the setup
cell first; every other cell runs top to bottom.

In [ ]:
# Setup: jammi 0.49.1 — the CUDA engine on an sm_80+ GPU (L4, A100, …), the
# CPU engine otherwise — and the cookbook's library and fixtures. The chapter runs
# at `small` scale, over the committed fixtures, in minutes. SCALE = "full" runs
# it over the published data and real encoders instead: meant for a GPU, and the
# chapters that fine-tune take hours there.
import os
import subprocess
import sys


def compute_capability() -> float:
    try:
        out = subprocess.run(
            ["nvidia-smi", "--query-gpu=compute_cap", "--format=csv,noheader"],
            capture_output=True, text=True, check=True,
        ).stdout.split()
    except (OSError, subprocess.CalledProcessError):
        return 0.0
    return float(out[0]) if out else 0.0


gpu = compute_capability() >= 8.0
engine = "jammi-ai-native-cu12" if gpu else "jammi-ai-native"
server = "jammi-server-cu12" if gpu else "jammi-server"
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "jammi-ai==0.49.1", engine + "==0.49.1", "jammi-cookbook==0.49.1"], check=True)
SCALE = "small"
os.environ["JAMMI_COOKBOOK_SCALE"] = SCALE
print(f"engine: {engine}   scale: {SCALE}")

In [ ]:
import jammi_cookbook

**Recipe:** `train_context_predictor` + `predict_with_context_predictor` +
`conformalize_interval` + `conformalize` · **Theory:** context-conditioned
posterior (Garnelo et al. 2018 CNP; Kim et al. 2019 ANP) + conformal coverage
and its break under graph dependence (Vovk; Angelopoulos & Bates 2021; Barber
et al. 2023; Tibshirani et al. 2019; CF-GNN, Huang et al. 2023; NAPS, Clarkson
et al. 2023) · **Rails:** provenance (which rows informed each prediction) +
measurement (coverage).

This is the tier Neptune structurally lacks: a **graph-conditioned
prediction** with an honest coverage guarantee. It carries two results:

1. **A regression-conformal workflow that runs end to end.** A Gaussian context
   predictor of a paper's *year* — a high-offset target (2014–2020) that
   collapses (std ≈ 0.001, mean ≈ 2163) unless the engine standardizes it.
   Writing this recipe found that collapse; the engine now standardizes the
   predictor's target and its context members' values in z-space and
   de-standardizes what it serves. The recipe runs because of that fix.
2. **The conformal-under-shift lesson.** Under the dataset's time split, both
   the year interval and the subject-classification set *under-cover*, and the
   textbook weighted-conformal remedy is a **no-op in both cases**, for two
   complementary reasons — each with the diagnostic that explains it.

Both are findings at `full` scale (4,000 papers, ModernBERT). At `small` scale
every cell below runs the same computation on the 400-paper core with a
random-weight encoder: the numbers check the pipeline and are frozen as
regression goldens, and the findings are asserted at `full` only.

In [ ]:
import tempfile

import jammi
import numpy as np
from jammi_cookbook import contracts, datasets, keystone, scale, shift

SCALE = scale.current()
ALPHA = 0.10  # ask for 90% coverage

db = jammi.connect(f"file://{tempfile.mkdtemp()}")
arxiv = datasets.arxiv(db, SCALE)
embeddings = keystone.embed(db, arxiv, SCALE)
propagated = keystone.propagate(db, arxiv, embeddings)

year = {
    r["paper_id"]: r["year"]
    for r in db.sql(f"SELECT paper_id, year FROM {arxiv.papers}.public.{arxiv.papers}").to_pylist()
}
cal_ids, test_ids = arxiv.split["valid"], arxiv.split["test"]
print(f"calibration era (2018): {len(cal_ids)} papers   test era (2019–): {len(test_ids)} papers")

## Part A — year regression, conformalized

The predictor learns, across subjects, to read a paper's year off the years of
its nearest papers — a Neural Process: one task per subject, the context
retrieved by embedding similarity at prediction time. It trains over the
propagated (citation-conditioned) embeddings and records that table: every
later prediction reads its context from the same vectors, whatever tables the
source gains afterwards.

In [ ]:
keystone.show(keystone.train_year_predictor)

Each prediction is one `predict_with_context_predictor` call:

In [ ]:
keystone.show(keystone.predict_years)

In [ ]:
predictor = keystone.train_year_predictor(db, arxiv, SCALE, propagated)
cal_mean, cal_std = keystone.predict_years(db, arxiv, predictor, cal_ids)
test_mean, test_std = keystone.predict_years(db, arxiv, predictor, test_ids)
cal_year = np.array([year[k] for k in cal_ids], dtype=float)
test_year = np.array([year[k] for k in test_ids], dtype=float)
pred_std = float(np.concatenate([cal_std, test_std]).mean())
print(f"predictor cal-era mean:  {cal_mean.mean():.2f}")
print(f"predictor test-era mean: {test_mean.mean():.2f}")
print(f"mean predictive std:     {pred_std:.3f}")

In [ ]:
contracts.assert_close("arxiv.tier04.reg_cal_mean", float(cal_mean.mean()), tol=1.0)
contracts.assert_close("arxiv.tier04.reg_test_mean", float(test_mean.mean()), tol=1.0)
contracts.assert_close("arxiv.tier04.reg_pred_std", pred_std, tol=0.5)
assert pred_std > 0.05  # standardized in z-space: the Gaussian head did not collapse

The interval comes from the engine: calibrate the absolute residual
$|y - \hat y|$ on the calibration era, apply its $1-\alpha$ quantile to the test
era.

In [ ]:
intervals = db.conformalize_interval(
    cal_mean.tolist(), cal_year.tolist(), test_mean.tolist(), alpha=ALPHA
)
reg_cov = float(np.mean([lo <= y <= hi for y, (lo, hi) in zip(test_year, intervals)]))
reg_width = float(np.mean([hi - lo for lo, hi in intervals]))
print(f"nominal coverage:    {1 - ALPHA:.2f}")
print(f"interval coverage:   {reg_cov:.3f}")
print(f"mean interval width: {reg_width:.2f} years")

In [ ]:
contracts.assert_close("arxiv.tier04.reg_interval_coverage", reg_cov, tol=0.04)
contracts.assert_close("arxiv.tier04.reg_interval_width", reg_width, tol=0.6)
if SCALE is scale.Scale.FULL:
    assert reg_cov < 1 - ALPHA  # the time split breaks exchangeability: under-coverage

At `full` scale the interval **under-covers**: realised coverage falls below
the nominal 0.90 — the later papers are not exchangeable with the earlier ones.

### Weighting the interval does not restore coverage

The textbook covariate-shift fix is importance-weighted conformal: reweight the
calibration residuals toward the test era before taking the quantile. It can
only help if the calibration papers most like the test era carry the larger
residuals the test era has. Whether they do is a property of the data, and the
correlation below measures it. At `full` scale the test era's residuals run
larger — a real spread shift — but within the calibration set residual size is
unrelated to test-era likeness, so reweighting barely moves the quantile. On
the small core the calibration papers most like the test era carry *smaller*
residuals, so reweighting toward them narrows the interval and loses coverage.

A calibration paper's **test-era likeness** is read off its neighbourhood: the
share of test-era papers among its nearest calibration- and test-era papers,
one query-by-example `search` each. Its odds estimate the paper's
test-to-calibration likelihood ratio, the weight weighted conformal needs.

In [ ]:
keystone.show(keystone.test_era_shares)

In [ ]:
shares = keystone.test_era_shares(db, arxiv, propagated, cal_ids, sizes=(10, 25, 50))
likeness = shares[keystone.NEIGHBOURS]

cal_resid = np.abs(cal_mean - cal_year)
test_resid = np.abs(test_mean - test_year)
wt_reg_cov = shift.residual_coverage(
    cal_resid, test_resid, weights=shift.density_ratio(likeness, keystone.NEIGHBOURS), alpha=ALPHA
)
reg_corr = float(np.corrcoef(cal_resid, likeness)[0, 1])
print(f"cal  mean |y−ŷ|: {cal_resid.mean():.3f}")
print(f"test mean |y−ŷ|: {test_resid.mean():.3f}")
print(f"weighted interval coverage:      {wt_reg_cov:.3f}   (Δ {wt_reg_cov - reg_cov:+.4f})")
print(f"corr(|residual|, test-likeness): {reg_corr:+.3f}")

In [ ]:
contracts.assert_close("arxiv.tier04.reg_cal_resid_mag", float(cal_resid.mean()), tol=0.25)
contracts.assert_close("arxiv.tier04.reg_test_resid_mag", float(test_resid.mean()), tol=0.25)
contracts.assert_close("arxiv.tier04.reg_weighted_coverage", wt_reg_cov, tol=0.04)
contracts.assert_close("arxiv.tier04.reg_resid_corr", reg_corr, tol=0.2)
if SCALE is scale.Scale.FULL:
    assert abs(wt_reg_cov - reg_cov) <= 0.02  # the reweight barely moves coverage

### The provenance rail

A prediction carries its audit trail: `source` says how its context was
assembled (`ann`, `edges`, `hybrid`) and `context_ref` names the rows that
informed it. Asked to condition on the citation neighbourhood instead of
embedding similarity, the same predictor reads its context off the graph:

In [ ]:
target = test_ids[0]
by_similarity = db.predict_with_context_predictor(
    predictor, source=arxiv.papers, target_key=target
)
by_citation = db.predict_with_context_predictor(
    predictor, source=arxiv.papers, target_key=target,
    edge_source=arxiv.cites, edge_src_column="src", edge_dst_column="dst",
    edge_direction="out", edge_hops=2,
)
for how, served in (("similarity", by_similarity), ("citations", by_citation)):
    print(f"{how:<10} source={served['source']:<6} context={len(served['context_ref'])} papers "
          f"→ {served['mean']:.1f} ± {served['std']:.2f}")

In [ ]:
assert by_similarity["source"] == "ann" and by_citation["source"] == "edges"
assert all(ref in year for ref in by_citation["context_ref"])

## Part B — subject classification under the same shift

The same lesson on a second task, with a complementary mechanism. The task is
subject classification; the predictor is a nearest-neighbour vote over the
**propagated** (citation-graph-conditioned, tier 02) embeddings: a paper's class
scores are the similarity-weighted subjects of its nearest training-era papers,
found by a query-by-example `search` filtered to the training era. It is
calibrated on 2018 and tested on 2019 onward.

In [ ]:
keystone.show(keystone.subject_scores)

In [ ]:
scores = keystone.subject_scores(db, arxiv, propagated)
accuracy = float(np.mean(scores.test_scores.argmax(1) == scores.test_labels))
print(f"{len(scores.classes)} classes; test-era accuracy: {accuracy:.3f}")

In [ ]:
contracts.assert_close("arxiv.tier04.classifier_accuracy", accuracy, tol=0.04)

### Marginal conformal under-covers on the graph

Marginal split-APS conformal (`conformalize`) takes the calibration scores,
finds the finite-sample $(1-\alpha)$ APS quantile, and forms a prediction *set*
per test paper. It is exactly valid **under exchangeability**.

In [ ]:
sets = db.conformalize(
    scores.cal_scores.tolist(), scores.cal_labels.tolist(), scores.test_scores.tolist(),
    alpha=ALPHA, score="aps",
)
marg_cov = float(np.mean([y in s for y, s in zip(scores.test_labels, sets)]))
marg_size = float(np.mean([len(s) for s in sets]))
print(f"engine APS marginal coverage: {marg_cov:.3f}   (mean set size {marg_size:.2f})")

In [ ]:
contracts.assert_close("arxiv.tier04.marginal_coverage", marg_cov, tol=0.03)
contracts.assert_close("arxiv.tier04.marginal_set_size", marg_size, tol=0.6)
if SCALE is scale.Scale.FULL:
    assert marg_cov < 1 - ALPHA

At `full` scale realised coverage falls **below** the nominal 0.90: a covariate
shift carried along the homophilous citation graph mis-calibrates the marginal
quantile.

### The textbook remedy moves coverage but does not restore it

Weighted split-conformal (Tibshirani et al. 2019) reweights the calibration
nonconformity toward the test era. To compare apples to apples,
`shift.aps_coverage` is one local APS routine used for both the marginal and the
weighted pass, so the only difference between them is the weights. Its
admission rule excludes the class that crosses q̂, a hair more conservative
than the engine's, so its marginal sits a little above the engine's; the
comparison that matters is between its own two passes.

In [ ]:
local_cov, _ = shift.aps_coverage(
    scores.cal_scores, scores.cal_labels, scores.test_scores, scores.test_labels,
    weights=None, alpha=ALPHA,
)
schemes = {f"knn-{size}": shift.density_ratio(share, size) for size, share in shares.items()}
deltas = {}
for name, weights in schemes.items():
    cov, size = shift.aps_coverage(
        scores.cal_scores, scores.cal_labels, scores.test_scores, scores.test_labels,
        weights=weights, alpha=ALPHA,
    )
    deltas[name] = cov - local_cov
    print(f"weighted ({name:<9}) coverage {cov:.3f}   Δ {cov - local_cov:+.4f}   set size {size:.2f}")
print(f"local marginal coverage          {local_cov:.3f}")

In [ ]:
contracts.assert_close("arxiv.tier04.local_marginal_coverage", local_cov, tol=0.03)
max_delta = max(abs(d) for d in deltas.values())
contracts.assert_close("arxiv.tier04.weighting_max_abs_delta", max_delta, tol=0.01)
if SCALE is scale.Scale.FULL:
    assert all(local_cov + d < 1 - ALPHA for d in deltas.values())  # none restores nominal
    assert min(deltas.values()) < 0  # not systematically toward nominal

At `full` scale the density ratio estimated over three neighbourhood sizes moves
coverage a little, **not consistently toward nominal**, and **none reaches
0.90**. Weighting nudges
coverage but cannot restore it here.

### Why: the shift is orthogonal to the conformal score

Weighted conformal can move the quantile only if test-era-like calibration
papers have systematically different nonconformity scores.

In [ ]:
nonconformity = shift.aps_nonconformity(scores.cal_scores, scores.cal_labels)
corr = float(np.corrcoef(nonconformity, likeness)[0, 1])
print(f"corr(nonconformity, test-likeness): {corr:+.3f}")

In [ ]:
contracts.assert_close("arxiv.tier04.score_shift_corr", corr, tol=0.12)
if SCALE is scale.Scale.FULL:
    assert abs(corr) < 0.25

At `full` scale the correlation is near zero: the time-split shift lies almost
entirely **orthogonal to the conformal score**, so reweighting the calibration
distribution toward the test era redistributes mass along a direction the
nonconformity barely depends on — the weighted quantile is essentially the
marginal one. Weighted conformal is not a universal patch: it repairs
under-coverage only when the shift is score-aligned (the conformal chapter
builds that case).

## The honest remedy: a governed calibration cohort

If a client-side reweight cannot restore coverage in *either* crux, what does?
A **governed, time-aware (or simply larger) calibration cohort**: choose the
calibration set so it is exchangeable with the deployment era — a Mondrian or
time-stratified cohort the consumer owns. The engine exposes the **marginal**
`conformalize*`; the cohort is the consumer's choice, exactly where the doctrine
puts it.

In [ ]:
db.close()

## Bridge note (a signature chapter — deepened in the bridge chapters)

> **A prediction is a context-conditioned posterior; honest coverage needs the
> consumer to own the cohort.** The context predictor is a Neural Process
> (Garnelo et al. 2018 CNP; Kim et al. 2019 ANP; the TabPFN/PFN line) — the
> context set is a `search`/walk over the citation graph. Conformal gives
> finite-sample coverage *under exchangeability* (Vovk; Angelopoulos & Bates
> 2021); under this dataset's time split both the regression interval and the
> classification set under-cover (Barber et al. 2023). The textbook remedy is
> weighted / stratified conformal (Tibshirani et al. 2019; NAPS, Clarkson et al.
> 2023; CF-GNN, Huang et al. 2023) — but it repairs only a shift the
> nonconformity score can *see*. This is the tier with **no monograph and no
> Neptune analogue** — the moat.